In [1]:
import torch
import torchvision.datasets
import torchvision.transforms

import torch.nn as nn
import torch.nn.functional as F
import torch.autograd.functional as AGF

import torch.optim as optim

import matplotlib.pyplot as plt

from math import prod
import numpy

In [2]:
class ShallowNet(nn.Module):
    def __init__(self, d_in, width, d_out, nonlin=F.relu):
      super(ShallowNet, self).__init__()
      self.width = width
      self.d_in = d_in
      self.d_out = d_out
      self.nonlin=nonlin
    
      self.in_linear = nn.Linear(d_in, width)
      self.out_linear = nn.Linear(width,d_out,bias=False)
      
    def forward(self, x):
      return self.out_linear(self.nonlin(self.in_linear(x)))


In [3]:
def simple_pruning(net,X,m):
    weights = net.nonlin(net.in_linear(X)).norm(dim=0)
    weights *= net.out_linear.weight.norm(dim=0)
    probs = weights / weights.sum()
    
    indices = torch.multinomial(probs,m,replacement=True)
    
    prunet = ShallowNet(net.d_in, m, net.d_out,net.nonlin)
    prunet.cuda()
    
    prunet.in_linear.weight.data = net.in_linear.weight[indices,:]
    prunet.in_linear.bias.data = net.in_linear.bias[indices]
    prunet.out_linear.weight.data = net.out_linear.weight[:,indices] / (m*probs[indices])
    return prunet

d = 50
net = ShallowNet(d,d*5,d)
net.cuda()
N = 500
X = torch.randn([N,d]).cuda()

prunet = simple_pruning(net,X,20000)

print(net(X).norm())
print(prunet(X).norm())
print((net(X)-prunet(X)).norm())


tensor(36.5541, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(36.7494, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
tensor(4.2135, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


In [7]:
def chained_pruning(net,X,m,K):
    prunX = torch.zeros_like(X).cuda()
    prev_alphas = torch.zeros_like(net.nonlin(net.in_linear(prunX)))
    weights = torch.ones([K,net.width]).cuda()
    thresh = (d**torch.linspace(0,1,K)).round().tolist()
    thresh = [0]+[int(i) for i in thresh]
    print(thresh)
    for k in range(K):
        prunX[:,:k+1] = X[:,:k+1]
        alphas = net.nonlin(net.in_linear(prunX))
        weights[k,:] = (alphas - prev_alphas).norm(dim=0) * net.out_linear.weight.norm(dim=0) / numpy.sqrt(thresh[k+1])
        
        prev_alphas = alphas
    
    probs = weights / weights.sum()
    ms = (torch.cumsum(probs.sum(dim=1),dim=0)*m).round().tolist()
    ms = [int(i) for i in ms]
    probs = probs / probs.sum(dim=1,keepdim=True)
    print(ms)
    
    prunet = ShallowNet(net.d_in, m*2, net.d_out,net.nonlin)
    prunet.cuda()
    prunet.in_linear.weight.data *= 0
    prunet.in_linear.bias.data *= 0
    prunet.out_linear.weight.data *= 0
    
    compute = 0 
    last_m = 0
    for k in range(K):
        mm = ms[k]-last_m
        compute += mm*thresh[k+1]
        if mm>0:
            indices = torch.multinomial(probs[k],mm,replacement=True)

            prunet.in_linear.weight.data[2*last_m:2*ms[k]:2,:] = net.in_linear.weight[indices,:]
            prunet.in_linear.weight.data[2*last_m+1:2*ms[k]+1:2,:] = net.in_linear.weight[indices,:]
            prunet.in_linear.weight.data[2*last_m:2*ms[k]:2,thresh[k]:] = 0
            prunet.in_linear.weight.data[2*last_m+1:2*ms[k]+1:2,thresh[k+1]:] = 0
            if k>0:
                prunet.in_linear.bias.data[2*last_m:2*ms[k]:2] = net.in_linear.bias[indices]
            prunet.in_linear.bias.data[2*last_m+1:2*ms[k]+1:2] = net.in_linear.bias[indices]

            prunet.out_linear.weight.data[:,2*last_m:2*ms[k]:2] = -net.out_linear.weight[:,indices] / (mm*probs[k,indices])
            prunet.out_linear.weight.data[:,2*last_m+1:2*ms[k]+1:2] = net.out_linear.weight[:,indices] / (mm*probs[k,indices])
        
        last_m = ms[k]
    print(compute)
    return prunet


d = 1000
net = ShallowNet(d,2500,1)
net.cuda()
N = 500
X = torch.randn([N,d]).cuda() / (1 + torch.arange(0,d).cuda())**2

prunet = chained_pruning(net,X,200000,6)
prunet2 = simple_pruning(net,X,10000)
print(d*10000)

print(net(X).norm().item(),prunet(X).norm().item(),prunet2(X).norm().item())
print((net(X)-prunet(X)).norm().item(), (net(X)-prunet2(X)).norm().item())

[0, 1, 4, 16, 63, 251, 1000]
[180354, 194971, 198586, 199566, 199890, 200000]
549726
10000000
0.12780612707138062 0.1311333030462265 0.19936200976371765
0.028104258701205254 0.07634645700454712
